# Speech → ISL Sign Gestures (for the robotic hand)
**Pipeline:** 🎤 speech → **Whisper** (pretrained, no training needed) → text → **trained ISL gesture model** → per-finger curl → **servo angles** → `gesture_table.h` for the ESP32.

What gets *trained* here:
1. **Hand-landmark extraction** from your ISL images with MediaPipe (21 points per hand).
2. **Logistic Regression** + **Neural network (MLP)** classifiers on those landmarks, with confidence scores (for the judges).
3. **Gesture templates**: the typical finger bend of every sign, learned from the data and converted to servo angles.

Runtime: *Runtime → Change runtime type → T4 GPU* (only Whisper and Plan B really need it). Run cells top to bottom.

## 1. Install & download the MediaPipe hand model

In [ ]:
!pip -q install mediapipe
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!ls -lh hand_landmarker.task

## 2. Get the dataset (Kaggle: *indian-sign-language-isl*, folder `Indian/`, classes 1–9 + A–Z)
**Before running:** click the 🔑 **Secrets** icon in Colab's left sidebar → *Add new secret* → Name `KAGGLE_API_TOKEN`, Value = your `KGAT_...` token → turn on **Notebook access**. Never paste the token into a code cell.
No token? Set `USE_KAGGLE = False` and point `ZIP_PATH` to a zip of the dataset in your Drive.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

USE_KAGGLE    = True
KAGGLE_SLUG   = 'prathumarikeri/indian-sign-language-isl'   # dataset used by the brsdincer notebook
ZIP_PATH      = '/content/drive/MyDrive/ISL/isl_dataset.zip' # only used if USE_KAGGLE = False
OUT           = '/content/drive/MyDrive/ISL/output'          # models + tables are saved here
MAX_PER_CLASS = 300      # images used per sign (~42k total exist; raise for accuracy, lower for speed)

import os, zipfile
os.makedirs(OUT, exist_ok=True)
if USE_KAGGLE:
    !pip -q install -U kagglehub
    import kagglehub
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')   # read from Colab Secrets
    WORK = kagglehub.dataset_download(KAGGLE_SLUG)
else:
    WORK = '/content/isl'
    if not os.path.exists(WORK):
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall(WORK)
print('Dataset at:', WORK)

IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def find_class_root(root):
    """Return the folder whose sub-folders (one per class) contain images."""
    best, best_n = root, 0
    for d, subdirs, _ in os.walk(root):
        n = 0
        for s in subdirs:
            try:
                if any(f.lower().endswith(IMG_EXT) for f in os.listdir(os.path.join(d, s))):
                    n += 1
            except PermissionError:
                pass
        if n > best_n:
            best, best_n = d, n
    return best

CLASS_ROOT = find_class_root(WORK)
CLASSES = sorted(c for c in os.listdir(CLASS_ROOT) if os.path.isdir(os.path.join(CLASS_ROOT, c)))
print('Class folder:', CLASS_ROOT)
print(len(CLASSES), 'classes:', CLASSES)

## 3. Extract hand landmarks + finger curls from every image

In [ ]:
import cv2, random, numpy as np, mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from tqdm.auto import tqdm

detector = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,                                  # many ISL signs use two hands
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3))

FINGERS = {'thumb': (1, 2, 3, 4), 'index': (5, 6, 7, 8), 'middle': (9, 10, 11, 12),
           'ring': (13, 14, 15, 16), 'pinky': (17, 18, 19, 20)}
FINGER_NAMES = list(FINGERS)

def joint_angle(a, b, c):
    v1, v2 = a - b, c - b
    cos = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

def finger_curls(h):
    """0 = finger straight, 1 = fully bent. h: (21,3) landmarks in pixels."""
    out = []
    for name, (m, p, d, t) in FINGERS.items():
        if name == 'thumb':
            bend = (180 - joint_angle(h[m], h[p], h[d])) + (180 - joint_angle(h[p], h[d], h[t]))
            out.append(np.clip(bend / 90, 0, 1))
        else:
            bend = ((180 - joint_angle(h[0], h[m], h[p])) +
                    (180 - joint_angle(h[m], h[p], h[d])) +
                    (180 - joint_angle(h[p], h[d], h[t])))
            out.append(np.clip(bend / 250, 0, 1))
    return np.array(out, np.float32)

def normalize(h):
    """Wrist at origin, scaled by wrist->middle-knuckle length => position/size invariant."""
    h = h - h[0]
    return h / (np.linalg.norm(h[9]) + 1e-6)

def extract(img_bgr):
    if min(img_bgr.shape[:2]) < 256:              # dataset images are small; MediaPipe likes bigger
        img_bgr = cv2.resize(img_bgr, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    H, W = img_bgr.shape[:2]
    rgb = np.ascontiguousarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    res = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    hands = [np.array([[p.x * W, p.y * H, p.z * W] for p in lms], np.float32) for lms in res.hand_landmarks]
    if not hands:
        return None, None, 0
    hands.sort(key=lambda h: h[0, 0])            # left-to-right in the image
    hands = hands[:2]
    feat = np.zeros((2, 21, 3), np.float32)      # slot 2 stays zeros for one-handed signs
    for i, h in enumerate(hands):
        feat[i] = normalize(h)
    primary = max(hands, key=lambda h: np.ptp(h[:, 1]))   # biggest hand drives the robot
    return feat.flatten(), primary, len(hands)

X, y, CURLS, NHANDS = [], [], [], []
for c in CLASSES:
    folder = os.path.join(CLASS_ROOT, c)
    files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(IMG_EXT)]
    random.seed(0); random.shuffle(files)
    files = files[:MAX_PER_CLASS]
    ok = 0
    for f in tqdm(files, desc=f'class {c}', leave=False):
        img = cv2.imread(f)
        if img is None:
            continue
        feat, primary, nh = extract(img)
        if feat is None:
            continue
        X.append(feat); y.append(c); CURLS.append(finger_curls(primary)); NHANDS.append(nh)
        ok += 1
    print(f'{c:>10}: hands found in {ok}/{len(files)} images')

X, y, CURLS, NHANDS = np.array(X), np.array(y), np.array(CURLS), np.array(NHANDS)
np.savez_compressed(f'{OUT}/landmarks.npz', X=X, y=y, curls=CURLS, nhands=NHANDS)
print('\nTotal samples:', X.shape, '-> saved to Drive (skip this cell next time and np.load it)')

> **Low detection rate?** If most classes show hands found in under ~50% of images (common for black-and-white / skin-segmented datasets), use **Plan B** at the bottom (CNN on raw images).

## 4. Train the classifiers (Logistic Regression + Neural Network)

In [ ]:
# To reuse saved landmarks later:  d = np.load(f'{OUT}/landmarks.npz'); X, y, CURLS, NHANDS = d['X'], d['y'], d['curls'], d['nhands']
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

counts = Counter(y)
keep = np.array([counts[c] >= 5 for c in y])          # drop classes with too few samples
dropped = sorted(c for c in counts if counts[c] < 5)
if dropped: print('Dropped (too few detected images):', dropped)
X, y, CURLS, NHANDS = X[keep], y[keep], CURLS[keep], NHANDS[keep]

le = LabelEncoder(); yi = le.fit_transform(y)
Xtr, Xte, ytr, yte = train_test_split(X, yi, test_size=0.2, stratify=yi, random_state=42)
print('train', Xtr.shape, ' test', Xte.shape, ' classes', len(le.classes_))

# --- Model A: Logistic Regression (simple, explainable, gives probabilities) ---
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, C=1.0))
logreg.fit(Xtr, ytr)
acc_lr = accuracy_score(yte, logreg.predict(Xte))
print(f'Logistic Regression test accuracy: {acc_lr:.3f}')

In [ ]:
# --- Model B: Neural network (MLP) ---
import tensorflow as tf
scaler = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)
n_cls = len(le.classes_)

mlp = tf.keras.Sequential([
    tf.keras.layers.Input((X.shape[1],)),
    tf.keras.layers.GaussianNoise(0.05),               # small augmentation
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(n_cls, activation='softmax')])
mlp.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist = mlp.fit(Xtr_s, ytr, validation_split=0.15, epochs=200, batch_size=64, verbose=0,
               callbacks=[tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)])
acc_mlp = mlp.evaluate(Xte_s, yte, verbose=0)[1]
print(f'MLP test accuracy: {acc_mlp:.3f}   (epochs run: {len(hist.history["loss"])})')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist.history['accuracy'], label='train'); ax[0].plot(hist.history['val_accuracy'], label='val'); ax[0].set_title('Accuracy'); ax[0].legend()
ax[1].plot(hist.history['loss'], label='train'); ax[1].plot(hist.history['val_loss'], label='val'); ax[1].set_title('Loss'); ax[1].legend()
plt.show()

In [ ]:
# --- Evaluation for the report / judges ---
pred = mlp.predict(Xte_s, verbose=0).argmax(1)
print(classification_report(yte, pred, target_names=le.classes_, digits=3))
fig, ax = plt.subplots(figsize=(max(8, n_cls * 0.35), max(8, n_cls * 0.35)))
ConfusionMatrixDisplay.from_predictions(yte, pred, display_labels=le.classes_, ax=ax,
                                        xticks_rotation=90, colorbar=False, cmap='Blues')
plt.title('MLP confusion matrix'); plt.tight_layout()
plt.savefig(f'{OUT}/confusion_matrix.png', dpi=150); plt.show()

print('\nSample predictions with confidence (top-3):')
for i in np.random.choice(len(Xte), 5, replace=False):
    p = mlp.predict(Xte_s[i:i+1], verbose=0)[0]
    top = p.argsort()[::-1][:3]
    print(f'true={le.classes_[yte[i]]:>6} | ' + ', '.join(f'{le.classes_[k]} {p[k]*100:.1f}%' for k in top))

In [ ]:
# --- Save models ---
import joblib, json
joblib.dump(logreg, f'{OUT}/isl_logreg.joblib')
joblib.dump(scaler, f'{OUT}/isl_scaler.joblib')
mlp.save(f'{OUT}/isl_mlp.keras')
json.dump(list(le.classes_), open(f'{OUT}/labels.json', 'w'))
tflite = tf.lite.TFLiteConverter.from_keras_model(mlp).convert()       # small model for laptop/Raspberry Pi
open(f'{OUT}/isl_mlp.tflite', 'wb').write(tflite)
print('Saved to', OUT, os.listdir(OUT))

## 5. Test on your own photo (good live demo for judges)

In [ ]:
up = files.upload()
for name in up:
    img = cv2.imread(name)
    feat, primary, nh = extract(img)
    if feat is None:
        print(name, '-> no hand detected'); continue
    p = mlp.predict(scaler.transform([feat]), verbose=0)[0]
    pl = logreg.predict_proba([feat])[0]
    top = p.argsort()[::-1][:3]
    print(name)
    print('  MLP   :', ', '.join(f'{le.classes_[k]} {p[k]*100:.1f}%' for k in top))
    print('  LogReg:', ', '.join(f'{le.classes_[k]} {pl[k]*100:.1f}%' for k in pl.argsort()[::-1][:3]))

## 5b. Live webcam test: click **Capture** (allow camera access). Re-run for another sign.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

JS = '''
async function takePhoto(quality) {
  const div = document.createElement('div');
  const btn = document.createElement('button'); btn.textContent = 'Capture';
  const video = document.createElement('video'); video.style.display = 'block';
  const stream = await navigator.mediaDevices.getUserMedia({video: true});
  div.appendChild(btn); div.appendChild(video); document.body.appendChild(div);
  video.srcObject = stream; await video.play();
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  await new Promise(r => btn.onclick = r);
  const c = document.createElement('canvas');
  c.width = video.videoWidth; c.height = video.videoHeight;
  c.getContext('2d').drawImage(video, 0, 0);
  stream.getVideoTracks()[0].stop(); div.remove();
  return c.toDataURL('image/jpeg', quality);
}'''

def take_photo(filename='photo.jpg', quality=0.9):
    display(Javascript(JS))
    data = eval_js(f'takePhoto({quality})')
    open(filename, 'wb').write(b64decode(data.split(',')[1]))
    return filename

def predict_image(path):
    img = cv2.imread(path)
    feat, primary, nh = extract(img)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off')
    if feat is None:
        plt.title('No hand detected - move closer / use better light'); plt.show(); return
    p  = mlp.predict(scaler.transform([feat]), verbose=0)[0]
    pl = logreg.predict_proba([feat])[0]
    k = p.argmax()
    plt.title(f'Prediction: {le.classes_[k]}  ({p[k]*100:.1f}%)'); plt.show()
    print('MLP   :', ', '.join(f'{le.classes_[i]} {p[i]*100:.1f}%' for i in p.argsort()[::-1][:3]))
    print('LogReg:', ', '.join(f'{le.classes_[i]} {pl[i]*100:.1f}%' for i in pl.argsort()[::-1][:3]))
    print('Servo angles:', templates[le.classes_[k]]['angles'] if 'templates' in globals() else '(run cell 6 first)')

predict_image(take_photo())

## 6. Learn gesture templates → servo angles for YOUR hand
For each sign, the median finger bend across all its images is turned into servo angles.
**Edit `SERVOS`** to match your PCA9685 wiring and calibrated open/closed angles. The defaults follow your build: thumb+index share one servo, plus middle, ring, pinky and a thumb-rotation servo.

In [ ]:
SERVOS = [   # channel on PCA9685, finger(s) that drive it, angle when finger open / fully closed
    {'name': 'thumb_index', 'channel': 0, 'fingers': ['thumb', 'index'], 'open': 0, 'closed': 180},
    {'name': 'middle',      'channel': 1, 'fingers': ['middle'],         'open': 0, 'closed': 180},
    {'name': 'ring',        'channel': 2, 'fingers': ['ring'],           'open': 0, 'closed': 180},
    {'name': 'pinky',       'channel': 3, 'fingers': ['pinky'],          'open': 0, 'closed': 180},
    {'name': 'thumb_rot',   'channel': 4, 'fingers': ['thumb'],          'open': 0, 'closed': 90},  # rough, tune by hand
]

def curls_to_angles(curl):
    angles = []
    for s in SERVOS:
        v = float(np.mean([curl[FINGER_NAMES.index(f)] for f in s['fingers']]))
        angles.append(int(round(s['open'] + v * (s['closed'] - s['open']))))
    return angles

templates = {'REST': {'curl': [0.0] * 5, 'angles': curls_to_angles(np.zeros(5)), 'two_handed': False}}
for c in le.classes_:
    m = (y == c)
    curl = np.median(CURLS[m], axis=0)
    templates[c] = {'curl': [round(float(v), 3) for v in curl],
                    'angles': curls_to_angles(curl),
                    'two_handed': bool((NHANDS[m] >= 2).mean() > 0.5)}

print(f'{"sign":>8} | ' + ' '.join(f'{s["name"]:>11}' for s in SERVOS) + ' | note')
for c, t in templates.items():
    print(f'{c:>8} | ' + ' '.join(f'{a:>11}' for a in t['angles']) + (' | TWO-HANDED (one robot hand can only approximate)' if t['two_handed'] else ' |'))

json.dump({'servos': SERVOS, 'gestures': templates}, open(f'{OUT}/gestures.json', 'w'), indent=2)

# C header for the ESP32 sketch
lines = ['// Auto-generated from the ISL dataset', '#pragma once', f'#define NUM_SERVOS {len(SERVOS)}',
         'const uint8_t SERVO_CHANNELS[NUM_SERVOS] = {' + ', '.join(str(s['channel']) for s in SERVOS) + '};',
         'struct Gesture { const char* label; uint8_t angles[NUM_SERVOS]; };',
         'const Gesture GESTURES[] = {']
lines += ['  {"%s", {%s}},' % (c, ', '.join(map(str, t['angles']))) for c, t in templates.items()]
lines += ['};', f'const int NUM_GESTURES = {len(templates)};']
open(f'{OUT}/gesture_table.h', 'w').write('\n'.join(lines))
print('\nSaved gestures.json and gesture_table.h to', OUT)

## 7. Speech → text → gesture sequence
Whisper is already trained on huge amounts of speech, so you **don't train** this part. Upload a short `.wav/.mp3/.m4a` of someone saying e.g. *"hello one two three"*.
This dataset has only 1–9 and A–Z, so *hello* is finger-spelled H-E-L-L-O (add a `hello/` folder of images later to get a single sign). Words that exist as a sign class are played as one gesture; numbers are converted to digits; any other word is **finger-spelled** letter by letter.

In [ ]:
import re, torch
from transformers import pipeline
asr = pipeline('automatic-speech-recognition', model='openai/whisper-small',
               device=0 if torch.cuda.is_available() else -1)

NUM_WORDS = {'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4', 'five': '5',
             'six': '6', 'seven': '7', 'eight': '8', 'nine': '9'}
LOOKUP = {c.lower(): c for c in templates}

def text_to_gestures(text):
    seq = []
    for word in re.findall(r'[a-z0-9]+', text.lower()):
        word = NUM_WORDS.get(word, word)
        if word in LOOKUP:                       # the whole word is a sign (e.g. 'hello', '5')
            seq.append(LOOKUP[word])
        else:                                    # finger-spell it
            seq += [LOOKUP[ch] for ch in word if ch in LOOKUP]
        seq.append('REST')                       # open hand between words
    return seq

up = files.upload()
audio = list(up)[0]
text = asr(audio, generate_kwargs={'language': 'english', 'task': 'transcribe'})['text']
print('Heard:', text)
for g in text_to_gestures(text):
    t = templates[g]
    print(f'{g:>8} -> servo angles {t["angles"]}' + ('  (two-handed sign, approximated)' if t['two_handed'] else ''))

## 8. Download everything

In [ ]:
!cd "$OUT" && zip -q -r /content/isl_outputs.zip . && ls -lh /content/isl_outputs.zip
files.download('/content/isl_outputs.zip')

---
## Plan B (if MediaPipe found too few hands): CNN on raw images. Works very well on this dataset
Gives a sign *classifier* with confidence scores, but can't produce finger-bend templates, so set the servo angles for each sign by hand.

In [ ]:
import tensorflow as tf
IMG = 128
train_ds = tf.keras.utils.image_dataset_from_directory(CLASS_ROOT, validation_split=0.2, subset='training',
                                                       seed=42, image_size=(IMG, IMG), batch_size=32)
val_ds = tf.keras.utils.image_dataset_from_directory(CLASS_ROOT, validation_split=0.2, subset='validation',
                                                     seed=42, image_size=(IMG, IMG), batch_size=32)
cnn_classes = train_ds.class_names
train_ds, val_ds = train_ds.prefetch(tf.data.AUTOTUNE), val_ds.prefetch(tf.data.AUTOTUNE)

base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False
inp = tf.keras.Input((IMG, IMG, 3))
x = tf.keras.layers.RandomRotation(0.05)(inp)          # no horizontal flip: left/right hand matters in ISL
x = tf.keras.layers.RandomZoom(0.1)(x)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
out = tf.keras.layers.Dense(len(cnn_classes), activation='softmax')(x)
cnn = tf.keras.Model(inp, out)
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.fit(train_ds, validation_data=val_ds, epochs=10)
cnn.save(f'{OUT}/isl_cnn.keras')